# 🔍 JobStreet Indonesia — Data Science Job Listing Scraper

**Pipeline**: Selenium (headless Chrome) → BeautifulSoup → Pandas → PySpark  
**Target**: `id.jobstreet.com` | Keyword: *Data Science* | Location: *Indonesia*  
**Output**: `raw_dataset_jobstreet.csv` → `processed_dataset_jobstreet.csv`

> ⚠️ Run cells **top-to-bottom** on a fresh Google Colab instance.

In [ ]:
# Installation
# Installs all required system packages and Python libraries for Google Colab.
import subprocess

print("[1/4] Installing Python packages ...")
subprocess.run(["pip", "install", "-q", "selenium", "beautifulsoup4", "pyspark", "webdriver-manager"], check=True)

print("[2/4] Installing Google Chrome ...")
subprocess.run(["wget", "-q", "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"], check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "./google-chrome-stable_current_amd64.deb"], check=True)

print("[3/4] Installing Java for PySpark ...")
subprocess.run(["apt-get", "install", "-y", "-q", "openjdk-17-jdk"], check=True)

# Verifikasi
r2 = subprocess.run(["google-chrome", "--version"], capture_output=True, text=True)
print("Chrome:", r2.stdout.strip())

print("Environment ready.")


[1/4] Installing Python packages ...
[2/4] Installing Google Chrome ...
[3/4] Installing Java for PySpark ...
Chrome: Google Chrome 149.0.7827.114
Environment ready.


In [ ]:
# Imports & Configuration
# All library imports and global config variables are defined here.

import os
import time
import re
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException,
)

# Configuration

KEYWORD              = "Data Science"
LOCATION             = "Indonesia"
MAX_PAGES            = 5
DELAY_BETWEEN_REQUESTS = 3          # seconds — polite crawl delay between detail pages
PAGE_LOAD_TIMEOUT    = 15           # seconds for WebDriverWait
RAW_OUTPUT           = "raw_dataset_jobstreet.csv"
PROCESSED_OUTPUT     = "processed_dataset_jobstreet.csv"

# JobStreet Indonesia search URL template.
# {keyword} → URL-encoded search term, {page} → 1-based page number.
BASE_SEARCH_URL = (
    "https://id.jobstreet.com/id/{keyword}-jobs"
    "?location={location}&sortmode=ListedDate&page={page}"
)

print("Imports & konfigurasi selesai.")
print(f"   Keyword  : {KEYWORD}")
print(f"   Location : {LOCATION}")
print(f"   Max Pages: {MAX_PAGES}")
print(f"   Raw CSV  : {RAW_OUTPUT}")
print(f"   Final CSV: {PROCESSED_OUTPUT}")

Imports & konfigurasi selesai.
   Keyword  : Data Science
   Location : Indonesia
   Max Pages: 5
   Raw CSV  : raw_dataset_jobstreet.csv
   Final CSV: processed_dataset_jobstreet.csv


In [ ]:
# Browser Setup
# Factory function that creates and returns a configured headless Chrome driver.

def setup_driver() -> webdriver.Chrome:
    """
    Configure and return a headless Chrome WebDriver instance suitable for
    running on Google Colab (Ubuntu, no display server).

    Chrome flags applied:
      - --headless          : Run without a visible window.
      - --no-sandbox        : Required inside Colab's container environment.
      - --disable-dev-shm-usage : Avoids /dev/shm size issues in Docker/Colab.
      - --window-size       : Ensures full-page layout (avoids mobile breakpoints).
      - Custom User-Agent   : Mimics a real desktop Chrome browser.

    Returns
    -------
    webdriver.Chrome
        A ready-to-use Chrome WebDriver instance.
    """
    options = Options()

    # Headless & sandbox settings
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-setuid-sandbox")

    # Viewport (ensures desktop layout)
    options.add_argument("--window-size=1920,1080")

    # Realistic User-Agent to reduce bot-detection likelihood
    options.add_argument(
        "user-agent=Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    # Additional stealth options
    options.add_argument("--lang=id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    # On Colab, chromium-chromedriver installs to /usr/bin/chromedriver
    options.binary_location = "/usr/bin/google-chrome"
    from webdriver_manager.chrome import ChromeDriverManager
    service = Service(ChromeDriverManager().install())

    driver = webdriver.Chrome(service=service, options=options)
    driver.implicitly_wait(5)

    print("WebDriver siap. Chromium berjalan dalam mode headless.")
    return driver


# Quick sanity-check (will be called again in the main pipeline cell)
print("ℹFungsi setup_driver() terdefinisi dan siap dipanggil.")

ℹFungsi setup_driver() terdefinisi dan siap dipanggil.


In [ ]:
# Search Page Scraper
# Extracts summary data (and detail URLs) from a single paginated result page.

def _safe_text(element, selector: str, attr: str | None = None, default: str = "Tidak tersedia") -> str:
    """
    Safely extract text or an attribute from a BeautifulSoup element.

    Parameters
    ----------
    element : bs4.element.Tag
        The parent BeautifulSoup tag to search within.
    selector : str
        A CSS selector string.
    attr : str or None
        If provided, return this HTML attribute value instead of text content.
    default : str
        Fallback value when the element or attribute is missing.

    Returns
    -------
    str
        Stripped text or attribute value, or *default* if not found.
    """
    try:
        tag = element.select_one(selector)
        if tag is None:
            return default
        if attr:
            return tag.get(attr, default).strip()
        return tag.get_text(separator=" ", strip=True) or default
    except Exception:
        return default


def scrape_search_page(driver: webdriver.Chrome, page_number: int) -> list[dict]:
    """
    Navigate to a JobStreet search results page and extract job card summaries.

    The function waits (up to PAGE_LOAD_TIMEOUT seconds) for at least one
    ``<article data-card-type="JobCard">`` element to appear before parsing.
    Fields extracted from each card:
      - job_title, company_name, location, job_type,
        salary_range, posted_date, detail_url

    Parameters
    ----------
    driver : webdriver.Chrome
        An active Selenium WebDriver instance.
    page_number : int
        1-based page index for the search results.

    Returns
    -------
    list[dict]
        A list of job summary dicts.  Returns an empty list on timeout or
        if no job cards are found.
    """
    keyword_slug = KEYWORD.lower().replace(" ", "-")
    location_slug = LOCATION.replace(" ", "%20")
    url = BASE_SEARCH_URL.format(
        keyword=keyword_slug,
        location=location_slug,
        page=page_number,
    )

    print(f"  → Navigasi ke: {url}")
    driver.get(url)

    # Wait for job cards
    try:
        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'article[data-card-type="JobCard"]')
            )
        )
    except TimeoutException:
        print(f"  ⚠️  Timeout — tidak ada JobCard ditemukan pada halaman {page_number}.")
        return []

    # Parse with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select('article[data-card-type="JobCard"]')

    if not cards:
        print(f"  ⚠️  Halaman {page_number}: tidak ada kartu lowongan ditemukan.")
        return []

    jobs = []
    for card in cards:
        # Job Title + Detail URL
        title_tag = (
            card.select_one('[data-automation="jobTitle"]')
            or card.select_one('a[data-automation="job-list-view-job-link"]')
            or card.select_one("h1, h2, h3")
        )
        job_title  = title_tag.get_text(strip=True) if title_tag else "Tidak tersedia"
        detail_href = title_tag.get("href", "") if title_tag else ""
        if detail_href and not detail_href.startswith("http"):
            detail_href = "https://id.jobstreet.com" + detail_href

        # Company Name
        company_name = _safe_text(card, '[data-automation="jobCompany"]')
        if company_name == "Tidak tersedia":
            company_name = _safe_text(card, '[data-automation="advertiser-name"]')

        # Location
        location_val = _safe_text(card, '[data-automation="jobCardLocation"]')
        if location_val == "Tidak tersedia":
            location_val = _safe_text(card, '[data-automation="job-card-location"]')

        # Job Type (work arrangement)
        job_type = _safe_text(card, '[data-automation="jobShortDescription"]')
        if job_type == "Tidak tersedia":
            job_type = _safe_text(card, '[data-automation="job-type"]')

        # Salary
        salary_range = _safe_text(card, '[data-automation="jobSalary"]')
        if salary_range == "Tidak tersedia":
            salary_range = _safe_text(card, '[data-automation="job-salary"]')

        # Posted Date
        posted_date = _safe_text(card, '[data-automation="jobListingDate"]')
        if posted_date == "Tidak tersedia":
            posted_date = _safe_text(card, 'time')
            if posted_date == "Tidak tersedia":
                time_tag = card.select_one("time")
                if time_tag:
                    posted_date = time_tag.get("datetime", "Tidak tersedia")

        jobs.append({
            "job_title"   : job_title,
            "company_name": company_name,
            "location"    : location_val,
            "job_type"    : job_type,
            "salary_range": salary_range,
            "posted_date" : posted_date,
            "detail_url"  : detail_href,
        })

    return jobs


print("Fungsi scrape_search_page() terdefinisi.")

Fungsi scrape_search_page() terdefinisi.


In [ ]:
# Detail Page Scraper

def _extract_section_text(soup: BeautifulSoup, heading_keywords: list) -> str:
    heading_tags = soup.find_all(["h2", "h3", "h4", "strong", "b"])
    for h in heading_tags:
        h_text = h.get_text(strip=True).lower()
        if any(kw.lower() in h_text for kw in heading_keywords):
            parts = []
            for sibling in h.find_next_siblings():
                if sibling.name in ["h2", "h3", "h4"]:
                    break
                text = sibling.get_text(separator="\n", strip=True)
                if text:
                    parts.append(text)
            if parts:
                return " | ".join(parts)
            parent_text = h.parent.get_text(separator=" ", strip=True)
            if parent_text and parent_text != h.get_text(strip=True):
                return parent_text
    return "Tidak tersedia"


def scrape_job_detail(driver: webdriver.Chrome, detail_url: str) -> dict:
    _default = {
        "job_type"           : "Tidak tersedia",
        "experience_level"   : "Tidak tersedia",
        "education_req"      : "Tidak tersedia",
        "job_requirements"   : "Tidak tersedia",
        "job_responsibilities": "Tidak tersedia",
    }

    if not detail_url:
        return _default

    try:
        driver.get(detail_url)

        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(
            EC.any_of(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, '[data-automation="jobAdDetails"]')
                ),
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, '[data-automation="job-detail-page"]')
                ),
            )
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # 1. job_type — dari elemen khusus di luar desc_block
        job_type_el = soup.select_one('[data-automation="job-detail-work-type"]')
        job_type = job_type_el.get_text(strip=True) if job_type_el else "Tidak tersedia"

        # 2. Job description block
        desc_block = (
            soup.select_one('[data-automation="jobAdDetails"]')
            or soup.select_one('[data-automation="job-detail-page"]')
            or soup.select_one(".job-description")
            or soup.find("div", class_=re.compile(r"jobAd|JobAd|job.?detail", re.I))
        )

        requirements_text     = "Tidak tersedia"
        responsibilities_text = "Tidak tersedia"
        edu                   = "Tidak tersedia"
        exp                   = "Tidak tersedia"

        if desc_block:
            full_text = desc_block.get_text(separator=" ", strip=True)

            # 3. Responsibilities
            responsibilities_text = _extract_section_text(
                desc_block,
                ["what you'll do", "responsibility", "responsibilities",
                 "tanggung jawab", "tugas", "deskripsi pekerjaan",
                 "job desc", "yang akan kamu lakukan", "kamu akan"],
            )

            # 4. Requirements
            requirements_text = _extract_section_text(
                desc_block,
                ["what we're looking for", "requirement", "requirements",
                 "kualifikasi", "persyaratan", "skill", "keahlian",
                 "yang kami cari", "must-have"],
            )
            if requirements_text == "Tidak tersedia":
                requirements_text = full_text[:2000] if full_text else "Tidak tersedia"

            # 5. Education
            edu = _extract_section_text(
                soup,
                ["Education", "Pendidikan", "education", "pendidikan",
                 "kualifikasi pendidikan", "educational requirement"],
            )

            # 6. Experience
            exp_match = re.search(
                r'(fresh\s*graduate[^.]{0,60}|'
                r'\d+\+?\s*[-–]\s*\d*\s*(tahun|year)[^.]{0,40}|'
                r'minimal?\s*\d+\s*(tahun|year)[^.]{0,40}|'
                r'pengalaman\s*\d+[^.]{0,40}|'
                r'experience\s*of\s*\d+[^.]{0,40})',
                full_text, re.IGNORECASE
            )
            exp = exp_match.group(0).strip() if exp_match else "Tidak tersedia"

        return {
            "job_type"           : job_type,
            "experience_level"   : exp,
            "education_req"      : edu,
            "job_requirements"   : requirements_text,
            "job_responsibilities": responsibilities_text,
        }

    except Exception as exc:
        print(f"Gagal: {detail_url} — {type(exc).__name__}: {exc}")
        return _default


print("Fungsi scrape_job_detail() terdefinisi.")

Fungsi scrape_job_detail() terdefinisi.


In [ ]:
# Main Collection Pipeline
# Orchestrates the full scraping run: search pages → detail pages → CSV save.

print("Pipeline started...")

driver = None
all_jobs: list[dict] = []

try:
    # Step 1: Inisialisasi WebDriver
    driver = setup_driver()

    # Step 2: Scrape halaman pencarian
    print()
    print("── FASE 1: Mengumpulkan daftar lowongan dari halaman pencarian ──")

    for page in range(1, MAX_PAGES + 1):
        print(f"\n[Halaman {page}/{MAX_PAGES}]")
        page_jobs = scrape_search_page(driver, page)
        all_jobs.extend(page_jobs)
        print(f" Halaman {page}: ditemukan {len(page_jobs)} lowongan "
              f"(total terkumpul: {len(all_jobs)})")

        if not page_jobs:
            print("Tidak ada data di halaman ini — kemungkinan halaman terakhir.")
            break

        # Short delay between search pages to be polite
        if page < MAX_PAGES:
            time.sleep(2)

    print(f"\nFase 1 selesai. Total lowongan ditemukan: {len(all_jobs)}")

    # Step 3: Scrape halaman detail
    print()
    print("── FASE 2: Mengambil detail setiap lowongan ──")

    total = len(all_jobs)
    for idx, job in enumerate(all_jobs, start=1):
        detail = scrape_job_detail(driver, job.get("detail_url", ""))
        job.update(detail)

        # Progress report every 10 jobs
        if idx % 10 == 0 or idx == total:
            print(f"  📋 Progress: {idx}/{total} lowongan selesai diproses")

        # Polite delay between detail page requests
        time.sleep(DELAY_BETWEEN_REQUESTS)

    print(f"\nFase 2 selesai. Semua {total} detail berhasil diproses.")

    # Step 4: Hapus kolom helper, tambah source_platform
    for job in all_jobs:
        job.pop("detail_url", None)          # remove internal helper column
        job["source_platform"] = "JobStreet" # add mandatory column

    # Step 5: Konversi ke DataFrame
    COLUMN_ORDER = [
        "job_title", "company_name", "location", "job_type",
        "experience_level", "education_req", "salary_range",
        "job_requirements", "job_responsibilities",
        "posted_date", "source_platform",
    ]

    df = pd.DataFrame(all_jobs)

    # Ensure all required columns exist (fill missing with "Tidak tersedia")
    for col in COLUMN_ORDER:
        if col not in df.columns:
            df[col] = "Tidak tersedia"

    df = df[COLUMN_ORDER]   # reorder to canonical order

    print()
    print("RINGKASAN DATASET")
    print(f"  Shape : {df.shape}")
    print(f"  Kolom : {list(df.columns)}")
    print()
    print(df.head(3).to_string())

    # Step 6: Simpan ke CSV
    df.to_csv(RAW_OUTPUT, index=False, encoding="utf-8-sig")
    print()
    print(f"Dataset mentah disimpan ke: {RAW_OUTPUT}")
    print(f"   Jumlah baris: {len(df)} | Jumlah kolom: {len(df.columns)}")

except KeyboardInterrupt:
    print("\nPipeline dihentikan oleh pengguna (KeyboardInterrupt).")
    if all_jobs:
        print(f"   Menyimpan {len(all_jobs)} data yang sudah terkumpul ...")
        pd.DataFrame(all_jobs).to_csv(RAW_OUTPUT, index=False, encoding="utf-8-sig")

except Exception as exc:
    print(f"\nPipeline error: {type(exc).__name__}: {exc}")
    raise

finally:
    if driver is not None:
        driver.quit()
        print("\n🔒 WebDriver ditutup dengan bersih.")

print()
print("Scraping phase finished")

Pipeline started...
WebDriver siap. Chromium berjalan dalam mode headless.

── FASE 1: Mengumpulkan daftar lowongan dari halaman pencarian ──

[Halaman 1/5]
  → Navigasi ke: https://id.jobstreet.com/id/data-science-jobs?location=Indonesia&sortmode=ListedDate&page=1
  ⚠️  Timeout — tidak ada JobCard ditemukan pada halaman 1.
 Halaman 1: ditemukan 0 lowongan (total terkumpul: 0)
Tidak ada data di halaman ini — kemungkinan halaman terakhir.

Fase 1 selesai. Total lowongan ditemukan: 0

── FASE 2: Mengambil detail setiap lowongan ──

Fase 2 selesai. Semua 0 detail berhasil diproses.

RINGKASAN DATASET
  Shape : (0, 11)
  Kolom : ['job_title', 'company_name', 'location', 'job_type', 'experience_level', 'education_req', 'salary_range', 'job_requirements', 'job_responsibilities', 'posted_date', 'source_platform']

Empty DataFrame
Columns: [job_title, company_name, location, job_type, experience_level, education_req, salary_range, job_requirements, job_responsibilities, posted_date, source_pla

In [ ]:
# PySpark Processing Pipeline
# Loads the raw CSV, runs data-quality checks, computes statistics, and saves

import os
import glob

# 1. Set JAVA_HOME before importing PySpark
# openjdk-17 is installed under /usr/lib/jvm/java-17-openjdk-amd64 on Ubuntu.
JAVA_HOME_PATH = "/usr/lib/jvm/java-17-openjdk-amd64"
if os.path.isdir(JAVA_HOME_PATH):
    os.environ["JAVA_HOME"] = JAVA_HOME_PATH
    print(f"JAVA_HOME diset ke: {JAVA_HOME_PATH}")
else:
    # Fallback: discover any installed JVM
    import subprocess as _sp
    result = _sp.run(["find", "/usr/lib/jvm", "-name", "java", "-type", "f"],
                     capture_output=True, text=True)
    if result.stdout.strip():
        fallback_java = result.stdout.strip().split("\n")[0]
        fallback_home = os.path.dirname(os.path.dirname(fallback_java))
        os.environ["JAVA_HOME"] = fallback_home
        print(f"JAVA_HOME (fallback) diset ke: {fallback_home}")
    else:
        print("Java tidak ditemukan! Pastikan Cell 1 dijalankan terlebih dahulu.")

# 2. Inisialisasi SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = (
    SparkSession.builder
    .appName("JobStreet_BigData_Pipeline")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession aktif. Versi Spark: {spark.version}")
print()

# 3. Load CSV sebagai Spark DataFrame
print(f"Memuat dataset mentah dari: {RAW_OUTPUT} ...")
sdf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(RAW_OUTPUT)
)

# 4. Schema
print("\n── SCHEMA ─────────────────────────────────────────────────────────")
sdf.printSchema()

# 5. Jumlah Baris
total_rows = sdf.count()
print(f"── TOTAL BARIS: {total_rows}")
print()

# 6. Ringkasan Non-Null per Kolom
REQUIRED_COLS = [
    "job_title", "company_name", "location", "job_type",
    "experience_level", "education_req", "salary_range",
    "job_requirements", "job_responsibilities",
    "posted_date", "source_platform",
]

print("── RINGKASAN NON-NULL PER KOLOM ────────────────────────────────────")
print(f"{'Kolom':<30} {'Non-Null':>10} {'Null':>10} {'% Terisi':>10}")
print("-" * 65)

for col_name in REQUIRED_COLS:
    if col_name in sdf.columns:
        non_null = sdf.filter(
            F.col(col_name).isNotNull() &
            (F.col(col_name) != "Tidak tersedia") &
            (F.col(col_name) != "")
        ).count()
        null_count = total_rows - non_null
        pct = (non_null / total_rows * 100) if total_rows > 0 else 0
        print(f"{col_name:<30} {non_null:>10} {null_count:>10} {pct:>9.1f}%")
    else:
        print(f"{col_name:<30} {'KOLOM TIDAK ADA':>30}")

print()

# 7. Top 10 Value Counts
for col_name, label in [
    ("location",     "LOKASI"),
    ("job_type",     "TIPE PEKERJAAN"),
    ("salary_range", "RENTANG GAJI"),
]:
    if col_name in sdf.columns:
        print(f"── TOP 10 {label} ───────────────────────────────────────────")
        (
            sdf.groupBy(col_name)
               .count()
               .orderBy(F.desc("count"))
               .limit(10)
               .show(truncate=50)
        )

# 8. Bersihkan baris di mana job_title null
before_clean = sdf.count()
sdf_clean = sdf.filter(
    F.col("job_title").isNotNull() & (F.col("job_title") != "")
)
after_clean = sdf_clean.count()
print(f"── PEMBERSIHAN: {before_clean - after_clean} baris dihapus "
      f"(job_title kosong). Sisa: {after_clean} baris.")
print()

# 9. Simpan hasil ke PROCESSED_OUTPUT
# Use a temp directory then rename to single file for ease of use
temp_output_dir = PROCESSED_OUTPUT + "_temp"

(
    sdf_clean
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temp_output_dir)
)

# Locate the single part file and rename
part_files = glob.glob(os.path.join(temp_output_dir, "part-*.csv"))
if part_files:
    import shutil
    shutil.move(part_files[0], PROCESSED_OUTPUT)
    shutil.rmtree(temp_output_dir, ignore_errors=True)
    print(f"File CSV tunggal disimpan ke: {PROCESSED_OUTPUT}")
else:
    print(f"Part file tidak ditemukan; direktori mentah ada di: {temp_output_dir}")

# 10. Selesai
print()
print(f"Pipeline selesai. File disimpan: {PROCESSED_OUTPUT}")

# 11. Stop SparkSession
spark.stop()
print("SparkSession ditutup.")

JAVA_HOME diset ke: /usr/lib/jvm/java-17-openjdk-amd64
SparkSession aktif. Versi Spark: 4.0.2

Memuat dataset mentah dari: raw_dataset_jobstreet.csv ...

── SCHEMA ─────────────────────────────────────────────────────────
root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- source_platform: string (nullable = true)

── TOTAL BARIS: 0

── RINGKASAN NON-NULL PER KOLOM ────────────────────────────────────
Kolom                            Non-Null       Null   % Terisi
-----------------------------------------------------------------
job_title                               0          0       0.0